# **2. Data processing**

Limpeza e padronização dos dados brutos para consumo pelas análises e visualizações.

## **3. Setup**

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'data' / 'raw' / 'manifest.csv').exists():
            return candidate
    raise FileNotFoundError('Não encontrei data/raw/manifest.csv no diretório atual ou nos pais.')


PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_MANIFEST_PATH = RAW_DATA_DIR / 'manifest.csv'
PROCESSED_MANIFEST_PATH = PROCESSED_DATA_DIR / 'manifest.csv'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'RAW_DATA_DIR: {RAW_DATA_DIR}')
print(f'PROCESSED_DATA_DIR: {PROCESSED_DATA_DIR}')

## **4. Cleaning rules**

As transformações são conservadoras: os dados de origem permanecem intactos em `raw`, enquanto `processed` recebe nomes consistentes, tipos previsíveis e somente campos informativos.

In [ ]:
COLUMN_RENAMES = {
    'freq': 'frequency',
    'geo_pict': 'geo_code',
    'time_period': 'year',
    'obs_value': 'value',
    'obs_status': 'observation_status',
    'unit_measure': 'unit',
    'error_val': 'error_value',
    'obs_comment': 'observation_comment',
    'climate_change_indicators': 'indicator',
}

GEO_NAMES = {
    'AS': 'American Samoa',
    'CK': 'Cook Islands',
    'FJ': 'Fiji',
    'FM': 'Federated States of Micronesia',
    'GU': 'Guam',
    'KI': 'Kiribati',
    'MH': 'Marshall Islands',
    'MP': 'Northern Mariana Islands',
    'NC': 'New Caledonia',
    'NR': 'Nauru',
    'NU': 'Niue',
    'PF': 'French Polynesia',
    'PG': 'Papua New Guinea',
    'PN': 'Pitcairn Islands',
    'PW': 'Palau',
    'SB': 'Solomon Islands',
    'TK': 'Tokelau',
    'TO': 'Tonga',
    'TV': 'Tuvalu',
    'VU': 'Vanuatu',
    'WF': 'Wallis and Futuna',
    'WS': 'Samoa',
}


def normalize_column_name(name: str) -> str:
    normalized = re.sub(r'[^a-z0-9]+', '_', name.strip().lower()).strip('_')
    return COLUMN_RENAMES.get(normalized, normalized)


def clean_dataset(slug: str) -> tuple[pd.DataFrame, dict]:
    source_path = RAW_DATA_DIR / slug / 'data.csv'
    frame = pd.read_csv(source_path, low_memory=False)
    rows_raw = len(frame)

    frame.columns = [normalize_column_name(column) for column in frame.columns]
    if frame.columns.duplicated().any():
        raise ValueError(f'{slug}: nomes de colunas duplicados após normalização')

    for column in frame.select_dtypes(include=['object', 'string']).columns:
        frame[column] = frame[column].astype('string').str.strip().replace('', pd.NA)

    empty_columns = [column for column in frame.columns if frame[column].isna().all()]
    frame = frame.drop(columns=empty_columns)

    for column in ['year', 'value', 'error_value']:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    if 'year' in frame.columns:
        frame['year'] = frame['year'].astype('Int64')

    if 'geo_code' in frame.columns:
        unknown_codes = sorted(set(frame['geo_code'].dropna()) - set(GEO_NAMES))
        if unknown_codes:
            raise ValueError(f'{slug}: códigos geográficos desconhecidos: {unknown_codes}')
        geo_position = frame.columns.get_loc('geo_code') + 1
        frame.insert(geo_position, 'geo_name', frame['geo_code'].map(GEO_NAMES))

    if slug == 'rainfall_anomalies' and 'unit' in frame.columns:
        unit_position = frame.columns.get_loc('unit') + 1
        frame.insert(unit_position, 'source_unit', frame['unit'])
        frame['unit'] = 'PERCENT_OF_1991_2020_NORMAL'

    required_columns = [column for column in ['geo_code', 'year', 'value'] if column in frame.columns]
    invalid_rows = int(frame[required_columns].isna().any(axis=1).sum())
    frame = frame.dropna(subset=required_columns)

    duplicates_removed = int(frame.duplicated().sum())
    frame = frame.drop_duplicates()

    sort_columns = [
        column
        for column in ['geo_code', 'agriculture_production_item', 'agriculture_production_type', 'year']
        if column in frame.columns
    ]
    frame = frame.sort_values(sort_columns, kind='stable').reset_index(drop=True)

    quality = {
        'slug': slug,
        'rows_raw': rows_raw,
        'rows_processed': len(frame),
        'duplicates_removed': duplicates_removed,
        'invalid_rows_removed': invalid_rows,
        'empty_columns_removed': '; '.join(empty_columns),
        'columns_processed': '; '.join(frame.columns),
        'output_file': f'{slug}.csv',
    }
    return frame, quality

## **5. Process all datasets**

In [ ]:
raw_manifest = pd.read_csv(RAW_MANIFEST_PATH)
quality_rows = []

for manifest_row in raw_manifest.to_dict('records'):
    slug = manifest_row['slug']
    processed, quality = clean_dataset(slug)
    output_path = PROCESSED_DATA_DIR / quality['output_file']
    processed.to_csv(output_path, index=False, float_format='%.10g')

    quality.update({
        'title': manifest_row['title'],
        'front': manifest_row['front'],
        'source_url': manifest_row['source_url'],
    })
    quality_rows.append(quality)

processed_manifest = pd.DataFrame(quality_rows)[[
    'slug',
    'title',
    'front',
    'rows_raw',
    'rows_processed',
    'duplicates_removed',
    'invalid_rows_removed',
    'empty_columns_removed',
    'columns_processed',
    'output_file',
    'source_url',
]]
processed_manifest.to_csv(PROCESSED_MANIFEST_PATH, index=False)

processed_readme = '''# Processed data

Analysis-ready tables generated by `notebooks/02-data_processing.ipynb`.

The pipeline standardizes column names and types, trims text, removes fully
empty columns, rejects rows without geography/year/value, removes exact
duplicates, adds territory names, and writes deterministic CSV files.

For rainfall anomalies, `unit` records the verified interpretation as a
percentage of the 1991–2020 normal; the source label is retained in
`source_unit`. See `manifest.csv` for row-level quality counts and lineage.
'''
(PROCESSED_DATA_DIR / 'README.md').write_text(processed_readme, encoding='utf-8')

display(processed_manifest)

## **6. Validation**

In [ ]:
assert len(processed_manifest) == len(raw_manifest)
assert (processed_manifest['rows_processed'] > 0).all()
assert processed_manifest['output_file'].map(lambda name: (PROCESSED_DATA_DIR / name).exists()).all()

rainfall = pd.read_csv(PROCESSED_DATA_DIR / 'rainfall_anomalies.csv')
assert rainfall['unit'].eq('PERCENT_OF_1991_2020_NORMAL').all()
assert rainfall[['geo_code', 'year', 'value']].notna().all().all()
assert not rainfall.duplicated().any()

display(processed_manifest[[
    'slug',
    'rows_raw',
    'rows_processed',
    'duplicates_removed',
    'invalid_rows_removed',
]])